In [ ]:
import pandas as pd
import geopandas as gpd

In [2]:
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union

def extract_polygon_geom(geom):
    
    """Extract Polygon/MultiPolygon geometries from a GeometryCollection"""

    if geom is None:
        return None
    
    elif geom.geom_type in ['Polygon', 'MultiPolygon']:
        return geom
    
    elif geom.geom_type == 'GeometryCollection':
        polygons = [g for g in geom.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]

        if not polygons:
            return None
        
        return unary_union(polygons)  # Merge into a MultiPolygon or Polygon
    
    else:
        return None  # Ignore Point, LineString, etc.

In [ ]:
def population_weighted_variables_mapping(
    shp_source_path, shp_target_path,
    data_csv, population_csv,
    source_id_col,
    target_id_col,
    variable_fields,
    population_field
):
    """
    Project variables from a source LSOA geometry (2001 or 2011) to 2021 LSOA geometry
    using population-weighted interpolation based entirely on spatial overlay.
    
    Parameters:
    - shp_source_path: path to source LSOA shapefile
    - shp_target_path: path to target LSOA shapefile
    - data_csv: CSV with variables to project
    - population_csv: CSV with population per source LSOA
    - source_id_col: column name for source LSOA ID
    - target_id_col: column name for target LSOA ID
    - variable_fields: list of variable fields to interpolate
    - population_field: name of population field in population_csv

    Returns:
    - DataFrame with [target_id_col] and interpolated variables
    """
    # Load shapefiles and data
    gdf_src = gpd.read_file(shp_source_path)
    gdf_tgt = gpd.read_file(shp_target_path)
    data_df = pd.read_csv(data_csv)
    pop_df = pd.read_csv(population_csv)

    # Clean column names
    data_df.columns = data_df.columns.str.replace('\xa0', ' ', regex=True).str.strip()
    pop_df.columns = pop_df.columns.str.replace('\xa0', ' ', regex=True).str.strip()

    # Keep necessary columns
    data_df = data_df[[source_id_col] + variable_fields]
    pop_df = pop_df[[source_id_col, population_field]]

    # Merge attributes into source GeoDataFrame
    gdf_src = gdf_src.merge(data_df, on=source_id_col, how='left')
    gdf_src = gdf_src.merge(pop_df, on=source_id_col, how='left')

    # Check required columns
    required_cols = [source_id_col, population_field] + variable_fields
    
    for col in required_cols:
        if col not in gdf_src.columns:
            raise KeyError(f"Missing column '{col}' in source GeoDataFrame.")

    # Ensure matching CRS
    gdf_tgt = gdf_tgt.to_crs(gdf_src.crs)

    # Compute source area
    gdf_src_clone = gdf_src.copy()
    gdf_src['area_total'] = gdf_src_clone.geometry.area

    # Spatial overlay — keep all geometries to avoid loss
    intersections = gpd.overlay(gdf_src, gdf_tgt, how='intersection', keep_geom_type=False)

    intersections['geometry'] = intersections.geometry.apply(extract_polygon_geom)
    intersections = intersections[intersections.geometry.notnull()]

    intersections['area_ij'] = intersections.geometry.area

    intersections['pop_ij'] = (intersections['area_ij'] / intersections['area_total']) * intersections[population_field]

    for field in variable_fields:
        intersections[field] = intersections[field].fillna(0)
        intersections[f'{field}_weighted'] = intersections['pop_ij'] / intersections[population_field] * intersections[field]

    agg_fields = {f'{field}_weighted': 'sum' for field in variable_fields}
    agg_fields['pop_ij'] = 'sum'

    result = intersections.groupby(target_id_col).agg(agg_fields).reset_index()

    result.drop(columns=['pop_ij'], inplace=True)

    return result

In [ ]:
df_num_of_sales = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Output/Number of sales.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["Number of sales 2011", "Number of sales 2021"],
    population_field = "All usual residents"
)

,LSOA21CD,Number of sales 2011_weighted,Number of sales 2021_weighted,pop_ij
0,E01000001,141.000000,123.000000,1464.999994
1,E01000002,233.000420,144.000238,1436.003369
2,E01000003,167.999580,94.999763,1345.996635
3,E01000005,41.000000,10.000000,984.999996
4,E01000006,44.000000,58.000000,1703.000000
...,...,...,...,...
4989,E01035718,189.000000,113.000000,3123.000000
4990,E01035719,74.641541,40.841598,992.403868
4991,E01035720,84.358463,46.158404,1121.596173
4992,E01035721,100.000010,89.000001,2451.000069


: 

In [ ]:
df_num_of_sales = population_weighted_variables_mapping(
    shp_source_path = "../Output/London_LSOA_2011.shp",
    shp_target_path = "../Output/London_LSOA_2021.shp",
    data_csv = "../Output/Number of sales.csv",
    population_csv = "../Output/Population_2011.csv",
    source_id_col = "LSOA11CD",
    target_id_col = "LSOA21CD",
    variable_fields = ["Number of sales 2011", "Number of sales 2021"],
    population_field = "All usual residents"
)